### Dim Customers

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimCustomers
WITH remove_dup as
(
select 
distinct (customer_id),
customer_name,
Customer_Name_Upper,
customer_email
from datamodeling.silver.silver_table
)
SELECT *,
 row_number() OVER (ORDER BY customer_id) as DimCustomerKey  
FROM remove_dup

num_affected_rows,num_inserted_rows


### Dim Products

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimProducts
WITH remove_dup as
(
SELECT 
DISTINCT (product_id),
product_name,
product_category
FROM datamodeling.silver.silver_table
)
SELECT *,
 row_number() OVER (ORDER BY product_id) as DimProductKey  
FROM remove_dup

num_affected_rows,num_inserted_rows


### Dim Payments

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimPayments
WITH remove_dup as
(
SELECT 
 DISTINCT(payment_type)
FROM datamodeling.silver.silver_table
)
SELECT *,
 row_number() OVER (ORDER BY payment_type) as DimPaymentKey  
FROM remove_dup

num_affected_rows,num_inserted_rows


### Dim Region

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimRegions
WITH remove_dup as
(
SELECT 
 DISTINCT(country)
FROM datamodeling.silver.silver_table
)
SELECT *,
 row_number() OVER (ORDER BY country) as DimRegionKey  
FROM remove_dup

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.dimregions

country,DimRegionKey
Canada,1
USA,2


### Dim Sales

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimSales
AS
SELECT 
 row_number() OVER (ORDER BY order_id) as DimSaleKey,
 order_id,
 order_date,
 customer_id,
 customer_name,
 customer_email,
 product_id,
 product_name,
 product_category,
 payment_type,
 country,
 last_updated,
 Customer_Name_Upper,
 processDate
FROM
  datamodeling.silver.silver_table

num_affected_rows,num_inserted_rows


### FACT TABLE

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.FactSales
AS
SELECT 
  S.DimSaleKey,
  C.DimCustomerKey,
  P.DimProductKey,
  R.DimRegionKey,
  PY.DimPaymentKey,
  F.quantity,
  F.unit_price
FROM 
  datamodeling.silver.silver_table F
LEFT JOIN 
  datamodeling.gold.dimcustomers C
  ON F.customer_id = C.customer_id
LEFT JOIN 
  datamodeling.gold.dimproducts P
  ON F.product_id = P.product_id
LEFT JOIN 
  datamodeling.gold.dimregions R
  ON F.country = R.country
LEFT JOIN 
  datamodeling.gold.dimpayments PY
  ON F.payment_type = PY.payment_type
LEFT JOIN 
  datamodeling.gold.dimsales S
  ON F.order_id = S.order_id


num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.factsales

DimSaleKey,DimCustomerKey,DimProductKey,DimRegionKey,DimPaymentKey,quantity,unit_price
1,1,1,2,1,1,999.99
2,2,2,2,2,2,199.99
3,3,3,1,1,1,129.99
4,4,4,2,1,1,899.99
5,1,3,2,1,2,129.99
